## Schritt 1: Setup und Pfade

Wir laden die Bibliotheken und legen fest, welche Dateien gelesen und geschrieben werden.

- `cv2` (OpenCV) liest und schreibt Videos und bearbeitet Bilder.
- `numpy` rechnet mit den Bildern (ein Bild ist ein Zahlen-Array).
- `matplotlib` zeigt Bilder an. Wir nutzen es nur für **bereits pixelierte** Frames.
- `csv`, `time`, `Path` brauchen wir für das Protokoll, die Zeitmessung und die Dateipfade.

Die Pfade sind **relativ zum Projektordner**: `Path.cwd().parent` ist der Ordner über `notebooks/`. So läuft derselbe Code auf dem Laptop und auf dem Klinik-PC, ohne dass du etwas ändern musst.

Die Ordnerstruktur trennt die Daten strikt:
- `data/raw/`: Rohvideo, wird **nie verändert**.
- `data/interim/`: Zwischenergebnisse (hier das Erkennungsprotokoll als CSV).
- `data/processed/`: Endergebnis (das pixelierte Video).

Alle drei Ordner stehen in `.gitignore`, die Videos landen also nicht auf GitHub.

Die Kontrollausgabe am Ende muss zweimal `True` zeigen. Bei `False` stimmt ein Pfad nicht.


In [8]:
import csv
import sys
import time
from collections import deque
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np

print("Python:", sys.executable)  # sollte auf .venv\Scripts\python.exe enden
print("OpenCV Version:", cv2.__version__)

# Pfade relativ zum Projektordner (das Notebook liegt in notebooks/).
PROJECT_ROOT = Path.cwd().parent
INPUT_VIDEO = PROJECT_ROOT / "data" / "raw" / "PAT_1" / "VID_76.AVI"
OUTPUT_VIDEO = PROJECT_ROOT / "data" / "processed" / "PAT_1" / "VID_76_yunet_pixelated.mp4"
LOG_CSV = PROJECT_ROOT / "data" / "interim" / "PAT_1" / "VID_76_detections.csv"
MODEL_PATH = PROJECT_ROOT / "models" / "face_detection_yunet_2023mar.onnx"

print("Video gefunden:", INPUT_VIDEO.exists())
print("Modell gefunden:", MODEL_PATH.exists())


Python: c:\Users\malin\Masterthesis\master-thesis\.venv\Scripts\python.exe
OpenCV Version: 5.0.0
Video gefunden: True
Modell gefunden: True


## Schritt 2: Parameter

- `SCORE_THRESHOLD`: Mindest-Sicherheit (0 bis 1), ab der YuNet eine Erkennung gelten lässt. Niedrigerer Wert findet mehr Gesichter, aber auch mehr Fehlalarme (z. B. Muster auf der Decke). Für Anonymisierung ist ein **verpasstes** Gesicht schlimmer als ein Fehlalarm, deshalb 0,3 statt 0,5.
- `N_BLOCKS`: Anzahl der Blöcke in der Breite des pixelierten Bereichs. **Kleiner = stärker anonymisiert.**
- `HOLD_FRAMES`: Wird in einem Frame kein Gesicht erkannt, pixelieren wir die letzte bekannte Box noch so viele Frames weiter. 15 Frames sind bei 25 fps etwa 0,6 Sekunden.
- `DETECT_SCALE`: Bei 1,0 sucht YuNet im Bild in voller Auflösung (1080p). Bei 0,5 wird das Bild vorher halbiert. Das ist schneller, aber kleine oder weit entfernte Gesichter werden eher verpasst.
- `MAX_FRAMES`: Begrenzt, wie viele Frames verarbeitet werden. Für den ersten Test 500 (ca. 20 Sekunden Video). Für das ganze Video auf `None` setzen.


In [9]:
SCORE_THRESHOLD = 0.3   # Mindest-Sicherheit der Erkennung. Niedrig = lieber Fehlalarm als verpasstes Gesicht
N_BLOCKS = 8            # Pixelation-Stärke (kleiner = stärker anonymisiert)
HOLD_FRAMES = 15        # so viele Frames wird die letzte Box weiterverwendet, wenn nichts erkannt wird
DETECT_SCALE = 1.0      # 1.0 = volle Auflösung (1080p). Bei zu langsam auf 0.5 setzen (schneller, aber kleine Gesichter eher verpasst)
MAX_FRAMES = 500        # erster Testlauf nur mit 500 Frames (ca. 20 s). Später auf None setzen für das ganze Video


## Schritt 3: Video-Metadaten lesen

Bevor wir verarbeiten, schauen wir, was für ein Video wir vor uns haben. Dabei wird **kein Bild** angezeigt, nur Zahlen.

- `cv2.VideoCapture` öffnet die Datei. `isOpened()` prüft, ob das geklappt hat (bei einem nicht unterstützten Format würde es scheitern).
- Mit `cap.get(...)` lesen wir Breite, Höhe, Bildrate (FPS) und die Anzahl der Frames.
- Dauer in Sekunden = Frames / FPS.
- `cap.release()` gibt die Datei wieder frei. Das machen wir immer, wenn wir fertig sind.

Erwartet für dieses Video: 1920 × 1080, 25 FPS, 8759 Frames, ca. 350 Sekunden.


In [10]:
cap = cv2.VideoCapture(str(INPUT_VIDEO))
if not cap.isOpened():
    raise RuntimeError(f"Video konnte nicht geöffnet werden: {INPUT_VIDEO}")

width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = cap.get(cv2.CAP_PROP_FPS)
n_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
cap.release()  # Datei wieder freigeben

print(f"Auflösung: {width} x {height}, FPS: {fps:.2f}, Frames: {n_frames}, Dauer: {n_frames / fps:.1f} s")


Auflösung: 1920 x 1080, FPS: 25.00, Frames: 8759, Dauer: 350.4 s


## Schritt 4: Pixelation eines Bildbereichs

Diese Funktion pixeliert **eine** Box `(x, y, Breite, Höhe)` in einem Frame und gibt das veränderte Bild zurück. Das Original bleibt unverändert (wir arbeiten auf einer Kopie).

Ablauf:
1. **Box vergrößern.** YuNet-Boxen sitzen eng auf dem Gesicht. Damit Haare, Ohren und Kinn mit verdeckt werden, geben wir links, rechts und unten 30 % Rand (`padding`) und oben 60 % (`padding_top`) zu. Dann schneiden wir die Box an den Bildrändern ab, sonst gäbe es Fehler bei Gesichtern am Rand.
2. **Ausschnitt verkleinern** auf `n_blocks` Pixel Breite (`INTER_AREA`). Jeder dieser Pixel ist der Durchschnitt eines Blocks.
3. **Wieder auf Originalgröße vergrößern**, ohne zu glätten (`INTER_NEAREST`). Jeder Pixel wird so zu einem sichtbaren Block.
4. Den Ausschnitt in die Kopie des Frames zurückschreiben.

Wichtig: Die Stärke hängt über `n_blocks` nur von der Blockanzahl ab, nicht von der Gesichtsgröße. Ein großes und ein kleines Gesicht sind damit gleich stark unkenntlich.


In [11]:
def pixelate_region(frame, box, n_blocks=8, padding=0.3, padding_top=0.6, inplace=False):
    """Pixeliert die Box (x, y, w, h) im Frame.

    padding:     Rand links/rechts/unten (Anteil von Breite bzw. Höhe)
    padding_top: größerer Rand oben, damit auch Haare/Scheitel bedeckt sind
    inplace:     True = das übergebene Bild direkt verändern (schneller, keine Kopie)
    """
    x, y, w, h = box
    img_h, img_w = frame.shape[:2]

    # Box vergrößern, danach an den Bildrändern abschneiden
    x1 = max(x - int(w * padding), 0)
    y1 = max(y - int(h * padding_top), 0)
    x2 = min(x + w + int(w * padding), img_w)
    y2 = min(y + h + int(h * padding), img_h)
    if x2 <= x1 or y2 <= y1:  # Box liegt komplett außerhalb des Bildes
        return frame if inplace else frame.copy()

    out = frame if inplace else frame.copy()
    roi = out[y1:y2, x1:x2]
    roi_h, roi_w = roi.shape[:2]

    # Verkleinern auf n_blocks Blöcke in der Breite (Höhe im gleichen Verhältnis, mind. 1)
    small_w = min(n_blocks, roi_w)
    small_h = max(1, round(roi_h * small_w / roi_w))
    small = cv2.resize(roi, (small_w, small_h), interpolation=cv2.INTER_AREA)

    # Wieder vergrößern, ohne zu glätten -> sichtbare Blöcke
    out[y1:y2, x1:x2] = cv2.resize(small, (roi_w, roi_h), interpolation=cv2.INTER_NEAREST)
    return out


## Schritt 5: Die ganze Pipeline als Funktion

`anonymize_video` fasst alles zusammen: Frame lesen, Gesichter erkennen, pixelieren, in ein neues Video schreiben. Weil es eine Funktion ist, kannst du sie später mit anderen Parametern oder anderen Videos wiederverwenden.

**Detektor erzeugen.** `cv2.FaceDetectorYN.create` lädt das YuNet-Modell. Die Bildgröße wird als `(Breite, Höhe)` angegeben. Achtung auf die Reihenfolge, sie ist anders als bei `frame.shape`. Mit `nms_threshold=0.3` werden überlappende Boxen zu einer zusammengefasst.

**Schleife über alle Frames.**
- Bei `DETECT_SCALE < 1` wird der Frame nur für die Erkennung verkleinert. Die Boxen rechnen wir danach wieder auf die volle Größe zurück (`/ detect_scale`). Pixeliert wird immer auf dem Originalframe.
- `detector.detect` liefert pro Gesicht eine Zeile mit Box, 5 Gesichtspunkten und Score. Wurde nichts gefunden, ist das Ergebnis `None`.

**Drei Zustände pro Frame**, die wir im Protokoll festhalten:
- `erkannt`: YuNet hat mindestens ein Gesicht gefunden. Wir merken uns die Boxen.
- `gehalten`: Nichts erkannt, aber vor weniger als `hold_frames` Frames gab es eine Box. Wir pixeln die alte Box weiter. Das überbrückt kurze Aussetzer, etwa bei Kopfdrehung oder Bewegungsunschärfe.
- `ungeschuetzt`: Nichts erkannt und keine alte Box mehr. Hier wird **nichts** pixeliert. Das sind die kritischen Frames.

**`try/finally`.** Egal ob ein Fehler auftritt: Video und Writer werden am Ende immer freigegeben. Sonst bleibt die Ausgabedatei unter Umständen kaputt oder gesperrt.

**Protokoll.** Pro Frame wird eine Zeile in `VID_76_detections.csv` gespeichert (Frame-Nummer, Anzahl Gesichter, bester Score, Status). Das brauchst du später, wenn du verschiedene Detektoren oder Einstellungen vergleichst.

**Rückgabe.** Die Funktion liefert eine Zusammenfassung: Anzahl Frames je Zustand, die Indizes der ungeschützten Frames, die längste Lücke ohne Schutz und die Laufzeit.

Hinweis: `mp4v` ist ein verlustbehafteter Codec, der ohne Zusatzsoftware funktioniert. Ton wird nicht übernommen, OpenCV schreibt nur Bild.


In [12]:
def anonymize_video(input_path, output_path, model_path, score_threshold=0.3,
                    n_blocks=8, hold_frames=15, detect_scale=1.0,
                    padding=0.3, padding_top=0.6, trail_frames=5,
                    debug=False, max_frames=None, log_csv=None):
    """Pixeliert alle Gesichter im Video. Gibt eine Zusammenfassung als Dict zurück.

    trail_frames: Boxen der letzten N Frames werden mitpixeliert (fängt Bewegung ab)
    debug:        True = Boxen und Scores ins AUSGABEvideo einzeichnen (nur zur Kontrolle!)
    """
    cap = cv2.VideoCapture(str(input_path))
    if not cap.isOpened():
        raise RuntimeError(f"Video konnte nicht geöffnet werden: {input_path}")

    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = cap.get(cv2.CAP_PROP_FPS)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    n_frames = total if max_frames is None else min(total, max_frames)

    # Der Detektor arbeitet auf der (evtl. verkleinerten) Bildgröße. Reihenfolge: (Breite, Höhe)
    det_w, det_h = int(width * detect_scale), int(height * detect_scale)
    detector = cv2.FaceDetectorYN.create(
        str(model_path), "", (det_w, det_h),
        score_threshold=score_threshold,
        nms_threshold=0.3,   # überlappende Boxen zusammenfassen
        top_k=5000,
    )

    output_path.parent.mkdir(parents=True, exist_ok=True)
    writer = cv2.VideoWriter(
        str(output_path), cv2.VideoWriter_fourcc(*"mp4v"), fps, (width, height)
    )

    rows = []              # Protokoll: eine Zeile pro Frame
    unprotected = []       # Indizes der Frames ohne jede Pixelation
    detected_frames = 0    # Frames mit frischer Erkennung
    held_frames = 0        # Frames mit weiterverwendeter alter Box
    last_boxes, age = [], 0
    recent = deque(maxlen=trail_frames + 1)  # Boxen der letzten Frames
    start = time.time()

    try:
        for i in range(n_frames):
            ok, frame = cap.read()
            if not ok:
                break

            # Zum Erkennen ggf. verkleinern (schneller), Boxen danach zurückrechnen
            if detect_scale == 1.0:
                small = frame
            else:
                small = cv2.resize(frame, (det_w, det_h), interpolation=cv2.INTER_AREA)
            _, faces = detector.detect(small)  # faces ist None, wenn nichts gefunden wurde

            if faces is not None:
                last_boxes = [(f[:4] / detect_scale).astype(int) for f in faces]
                n_faces, best_score = len(faces), float(faces[:, -1].max())
                age = 0
                detected_frames += 1
                status = "erkannt"
            else:
                n_faces, best_score = 0, 0.0
                age += 1
                if last_boxes and age <= hold_frames:
                    held_frames += 1
                    status = "gehalten"
                else:
                    last_boxes = []
                    status = "ungeschuetzt"

            # Boxen der letzten Frames mitnehmen: fängt schnelle Kopfbewegungen ab
            recent.append(list(last_boxes))
            boxes_to_blur = [b for boxes in recent for b in boxes]
            if status == "ungeschuetzt":
                if boxes_to_blur:
                    status = "nachgezogen"   # vom Nachlauf der letzten Frames noch gedeckt
                else:
                    unprotected.append(i)

            out = frame.copy()  # nur EINE Kopie pro Frame
            for box in boxes_to_blur:
                pixelate_region(out, box, n_blocks=n_blocks, padding=padding,
                                padding_top=padding_top, inplace=True)

            # Nur zur Kontrolle: erkannte Boxen mit Score ins Ausgabevideo zeichnen
            if debug and faces is not None:
                for f in faces:
                    x, y, w, h = (f[:4] / detect_scale).astype(int)
                    cv2.rectangle(out, (x, y), (x + w, y + h), (0, 255, 0), 2)
                    cv2.putText(out, f"{f[-1]:.2f}", (x, max(y - 5, 15)),
                                cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)

            writer.write(out)
            rows.append((i, n_faces, round(best_score, 3), status))
    finally:
        cap.release()   # egal was passiert: Dateien freigeben
        writer.release()

    # Protokoll als CSV speichern (Grundlage für die spätere Auswertung)
    if log_csv is not None:
        log_csv.parent.mkdir(parents=True, exist_ok=True)
        with open(log_csv, "w", newline="", encoding="utf-8") as f:
            w = csv.writer(f)
            w.writerow(["frame", "n_faces", "best_score", "status"])
            w.writerows(rows)

    # Längste Lücke ohne Schutz (aufeinanderfolgende Frame-Indizes)
    longest_gap, run = 0, 0
    for k, idx in enumerate(unprotected):
        run = run + 1 if k > 0 and idx == unprotected[k - 1] + 1 else 1
        longest_gap = max(longest_gap, run)

    return {
        "frames": len(rows),
        "erkannt": detected_frames,
        "gehalten": held_frames,
        "nachgezogen": sum(1 for r in rows if r[3] == "nachgezogen"),
        "ungeschuetzt": len(unprotected),
        "laengste_luecke": longest_gap,
        "ungeschuetzte_frames": unprotected,
        "sekunden": round(time.time() - start, 1),
    }


## Schritt 6: Pipeline ausführen

Wir rufen die Funktion mit den Parametern aus Schritt 2 auf und geben eine Zusammenfassung aus.

So liest du die Zahlen:
- **mit Erkennung:** Frames, in denen YuNet selbst ein Gesicht gefunden hat. Das sollte der Großteil sein.
- **mit gehaltener Box:** Kurze Aussetzer, die wir überbrückt haben. Viele davon deuten auf unsichere Erkennung hin.
- **ohne Schutz:** Frames, in denen nichts pixeliert wurde. **Das Ziel ist 0.** Jeder dieser Frames kann ein erkennbares Gesicht zeigen.
- **längste Lücke:** Wie viele Frames am Stück ungeschützt waren. Ein einzelner Frame ist weniger kritisch als zwei Sekunden am Stück.

Beginne mit `MAX_FRAMES = 500`. Wenn die Laufzeit und die Zahlen sinnvoll aussehen, setze `MAX_FRAMES = None` und führe Schritt 2 und 6 für das ganze Video erneut aus.


In [13]:
DEBUG_VIDEO = PROJECT_ROOT / "data" / "processed" / "PAT_1" / "VID_76_debug.mp4"

result = anonymize_video(
    INPUT_VIDEO, DEBUG_VIDEO, MODEL_PATH,
    score_threshold=SCORE_THRESHOLD,
    n_blocks=N_BLOCKS,
    hold_frames=HOLD_FRAMES,
    detect_scale=DETECT_SCALE,
    padding=0.5,          # etwas größerer Rand als vorher (0.3)
    padding_top=0.8,
    trail_frames=5,
    debug=True,           # Boxen + Scores einzeichnen
    max_frames=MAX_FRAMES,
    log_csv=LOG_CSV,
)

print(f"Frames: {result['frames']} (Dauer: {result['sekunden']} s)")
print(f"  erkannt: {result['erkannt']} | gehalten: {result['gehalten']} | nachgezogen: {result['nachgezogen']}")
print(f"  ohne Schutz: {result['ungeschuetzt']} (längste Lücke: {result['laengste_luecke']} Frames)")


Frames: 500 (Dauer: 140.4 s)
  erkannt: 487 | gehalten: 13 | nachgezogen: 0
  ohne Schutz: 0 (längste Lücke: 0 Frames)
